In [1]:
%pip install --upgrade fastapi uvicorn openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Creating a Simple API
from fastapi import FastAPI

app=FastAPI()

@app.get("/")
def home():
    return {"message":"My first API is running."}

In [3]:
#Building the LLM function
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found.")

client = OpenAI(api_key=api_key)


def generate_answer(question: str) -> str:
    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=(
            "You are a helpful AI assistant. "
            "Answer clearly and concisely."
        ),
        input=question,
        max_output_tokens=250
    )

    return response.output_text

In [4]:
answer = generate_answer("Explain RAG in simple English.")
print(answer)

RAG stands for "Retrieval-Augmented Generation." It is a method used in artificial intelligence that helps computers give better answers.

Here’s how it works simply:

1. **Retrieval:** The system first looks for useful information from a large set of documents or data.
2. **Augmented Generation:** Then, it uses that information to help create or generate a smart, accurate response.

So, instead of only using what it has learned before, the AI finds extra facts to give better answers.


In [5]:
#Creating the /ask API endpoint
from pydantic import BaseModel, Field
from fastapi import HTTPException


class QuestionRequest(BaseModel):
    question: str = Field(
        min_length=1,
        max_length=2000,
        description="The question to ask the AI assistant"
    )


@app.post("/ask")
def ask_ai(request: QuestionRequest):
    try:
        answer = generate_answer(request.question)

        return {
            "question": request.question,
            "answer": answer
        }

    except Exception:
        raise HTTPException(
            status_code=500,
            detail="The AI service could not process your request."
        )

In [6]:
#The notebook helped us understand the code. Now combine the pieces into one application file.